In [170]:
# Importing torch
import numpy as np
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.13.0+cu126
0.28.0+cu126


In [171]:
# GPU avaliability
if torch.cuda.is_available():
    print("GPU is available!")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU not available. Using CPU.")

GPU is available!
Using GPU: NVIDIA GeForce MX350


#### Tensor Creation Methods

In [172]:
# This method will assign a memory to create a tensor of provided shape and return the existing numbers at that location
a = torch.empty(size=[2, 3], dtype=torch.float) # default float32
a

tensor([[0.0266, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000]])

Scaler (1D) → Vector (2D) → Tensor (> 2D)

In [173]:
# Check type
type(a)

torch.Tensor

In [174]:
# Check the memory consuption
from sys import getsizeof

print(f"Total Memory {getsizeof(a)}")
print(f"Tensor Memory {a.nbytes}") # Float of 4 bytes

Total Memory 72
Tensor Memory 24


*The core difference is that sys.getsizeof() measures the total memory footprint of an object (including overhead and references), while .nbytes measures only the raw data size of the object's elements, excluding overhead.*

In [175]:
# Using zeros
torch.zeros(size = [2, 3], dtype = torch.float32)

tensor([[0., 0., 0.],
        [0., 0., 0.]])

In [176]:
# using ones
torch.ones(size = [2, 3])

tensor([[1., 1., 1.],
        [1., 1., 1.]])

In [177]:
# using rand
torch.rand(size = [2, 3])

tensor([[0.0177, 0.7999, 0.3203],
        [0.1841, 0.5128, 0.7848]])

In [178]:
# manual_seed - for regenerating the same array
seed_number = torch.initial_seed() # Store this seed number and use it as a variable → python long datatype
print(seed_number)
# NOTE: seed number act as a starting point to generate the sequences and numbers avalable on those locations will be in the resultant tensor

torch.manual_seed(seed_number)
torch.rand(2,3)

90934288176848358


tensor([[0.1560, 0.9888, 0.9006],
        [0.3385, 0.8168, 0.5418]])

In [179]:
# Device agnostic code
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [180]:
# using tensor

torch.tensor(
    data = [[1,2,3],[4,5,6]],
    dtype = torch.float32,
    requires_grad = False,
    device = device,
    pin_memory = True # The tensor will directly loded into pin-memory and not pager-memory, for faster loading. pin_memory=True is primarily a GPU optimization tool that prepares CPU data for faster transfer to the GPU.
)

tensor([[1., 2., 3.],
        [4., 5., 6.]], device='cuda:0')

In [181]:
# NOTE: You can only calculate the gradient when the end result (loss function) is scaler.
try:
    a = torch.rand(size=[2, 3], requires_grad=True, pin_memory=True).to(device=device)
    b = a.pow(exponent=2)
    c = b * 2 # c = (a ^ 2) + 2

    c.backward()

except RuntimeError as e:
    print(e)

grad can be implicitly created only for scalar outputs


In [182]:
# Question: Which one is fast?
torch.rand(size=[2, 3]).to(device="cuda") # Normal
torch.rand(size=[2, 3], pin_memory=True).to(device="cuda") # Slow: CPU (pager memory) → CPU (pin memory) → VRAM(GPU memory)
torch.rand(size=[2, 3], device="cuda") # Fast

tensor([[0.4120, 0.5870, 0.2428],
        [0.4865, 0.2196, 0.3961]], device='cuda:0')

[A guide on good usage of non_blocking and pin_memory() in PyTorch](https://docs.pytorch.org/tutorials/intermediate/pinmem_nonblock.html)

In [183]:
# arange
print("using arange ->", torch.arange(start=0, end=10, step=2))

# using linspace
print("using linspace ->", torch.linspace(start=0, end=10, steps=10))

# using eye - Identity Matrix tensor
print("using eye ->", torch.eye(n=5))

# using full - torch.ones(size = (m, n)) * constant( = 5)
print("using full ->", torch.full(size=(3, 3), fill_value=5))

using arange -> tensor([0, 2, 4, 6, 8])
using linspace -> tensor([ 0.0000,  1.1111,  2.2222,  3.3333,  4.4444,  5.5556,  6.6667,  7.7778,
         8.8889, 10.0000])
using eye -> tensor([[1., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 1.]])
using full -> tensor([[5, 5, 5],
        [5, 5, 5],
        [5, 5, 5]])


---

#### Tensor Shape

In [184]:
x = torch.tensor([[1,2,3],[4,5,6]])
x

tensor([[1, 2, 3],
        [4, 5, 6]])

In [185]:
print(x.shape)
print(x.size(dim=1)) # You can pass the dimention as well → dim = 0 (rows - outermost dimention) 

torch.Size([2, 3])
3


In [186]:
# Create an uninitialized tensor with the same shape as the input tensor
y = torch.empty_like(
    input=x,
    memory_format=torch.preserve_format, # The individual elements in this tensor pointing to the same location where the elements of first tensor is pointing -> else some random numbers with equal shape 
    dtype=torch.float32,
    device=device,
    pin_memory=False,
    requires_grad=True
)

print(y)

tensor([[0.4120, 0.5870, 0.2428],
        [0.4865, 0.2196, 0.3961]], device='cuda:0', requires_grad=True)


In [187]:
# torch.preserve_format
print(id(x[0][0]))
print(id(y[0][0]))
# NOTE: Since my device is changed (x - cpu) and (y - gpu) the memory locations of elements are different.

139244370480592
139244370481744


In [188]:
torch.zeros_like(x)

tensor([[0, 0, 0],
        [0, 0, 0]])

In [189]:
torch.ones_like(x)

tensor([[1, 1, 1],
        [1, 1, 1]])

In [190]:
torch.rand_like(x, dtype=torch.float32)

tensor([[0.9063, 0.7875, 0.5038],
        [0.8485, 0.1474, 0.4358]])

*`_like()` These methods create a new tensor with the exact same shape, dtype, and device as an existing input tensor.*

---

#### Tensor DataTypes

In [191]:
# find data type
x.dtype

torch.int64

In [192]:
# assign data type
torch.tensor([1.0, 2.0, 3.0], dtype = torch.int32)

tensor([1, 2, 3], dtype=torch.int32)

In [193]:
torch.tensor([1, 2, 3], dtype = torch.float64)

tensor([1., 2., 3.], dtype=torch.float64)

In [194]:
# using to() - Type Conversion - Generally used to transfer a tensor from cpu to gpu
x.to(torch.float32)

tensor([[1., 2., 3.],
        [4., 5., 6.]])

| **Data Type**             | **Dtype**         | **Description**                                                                                                                                                                |
|---------------------------|-------------------|--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| **32-bit Floating Point** | `torch.float32`   | Standard floating-point type used for most deep learning tasks. Provides a balance between precision and memory usage.                                                         |
| **64-bit Floating Point** | `torch.float64`   | Double-precision floating point. Useful for high-precision numerical tasks but uses more memory.                                                                               |
| **16-bit Floating Point** | `torch.float16`   | Half-precision floating point. Commonly used in mixed-precision training to reduce memory and computational overhead on modern GPUs.                                            |
| **BFloat16**              | `torch.bfloat16`  | Brain floating-point format with reduced precision compared to `float16`. Used in mixed-precision training, especially on TPUs.                                                |
| **8-bit Floating Point**  | `torch.float8`    | Ultra-low-precision floating point. Used for experimental applications and extreme memory-constrained environments (less common).                                               |
| **8-bit Integer**         | `torch.int8`      | 8-bit signed integer. Used for quantized models to save memory and computation in inference.                                                                                   |
| **16-bit Integer**        | `torch.int16`     | 16-bit signed integer. Useful for special numerical tasks requiring intermediate precision.                                                                                    |
| **32-bit Integer**        | `torch.int32`     | Standard signed integer type. Commonly used for indexing and general-purpose numerical tasks.                                                                                  |
| **64-bit Integer**        | `torch.int64`     | Long integer type. Often used for large indexing arrays or for tasks involving large numbers.                                                                                  |
| **8-bit Unsigned Integer**| `torch.uint8`     | 8-bit unsigned integer. Commonly used for image data (e.g., pixel values between 0 and 255).                                                                                    |
| **Boolean**               | `torch.bool`      | Boolean type, stores `True` or `False` values. Often used for masks in logical operations.                                                                                      |
| **Complex 64**            | `torch.complex64` | Complex number type with 32-bit real and 32-bit imaginary parts. Used for scientific and signal processing tasks.                                                               |
| **Complex 128**           | `torch.complex128`| Complex number type with 64-bit real and 64-bit imaginary parts. Offers higher precision but uses more memory.                                                                 |
| **Quantized Integer**     | `torch.qint8`     | Quantized signed 8-bit integer. Used in quantized models for efficient inference.                                                                                              |
| **Quantized Unsigned Integer** | `torch.quint8` | Quantized unsigned 8-bit integer. Often used for quantized tensors in image-related tasks.                                                                                     |


---

#### Mathematical Operations

In [195]:
x = torch.rand(2,2)
x

tensor([[0.8167, 0.4472],
        [0.4303, 0.6292]])

In [196]:
# Scalar Operations

# addition
print(x + 2, end = "\n\n")

# substraction
print(x - 2, end = "\n\n")

# multiplication
print(x * 3, end = "\n\n")

# division
print(x / 3, end = "\n\n")

# int division
print((x * 100) // 3, end = "\n\n")

# mod
print(((x * 100) // 3) % 2, end = "\n\n")

# power
print(x ** 2)

tensor([[2.8167, 2.4472],
        [2.4303, 2.6292]])

tensor([[-1.1833, -1.5528],
        [-1.5697, -1.3708]])

tensor([[2.4501, 1.3416],
        [1.2909, 1.8876]])

tensor([[0.2722, 0.1491],
        [0.1434, 0.2097]])

tensor([[27., 14.],
        [14., 20.]])

tensor([[1., 0.],
        [0., 0.]])

tensor([[0.6670, 0.2000],
        [0.1852, 0.3959]])


In [197]:
# floor division
(torch.tensor([1.0, 2.0, 3.0]) * 100) // 3

tensor([ 33.,  66., 100.])

In [198]:
# Element wise operation
a = torch.rand(2,3)
b = torch.rand(2,3)

print(a)
print(b)

tensor([[0.3581, 0.6317, 0.5861],
        [0.5851, 0.2806, 0.4553]])
tensor([[0.6674, 0.2621, 0.0398],
        [0.0518, 0.4979, 0.0048]])


In [199]:
# Shape must be same or able to broadcast

# add
print(a + b, end="\n\n")

# sub
print(a - b, end="\n\n")

# multiply
print(a * b, end="\n\n")

# division
print(a / b, end="\n\n")

# power
print(a ** b, end="\n\n")

# mod
print(a % b)

tensor([[1.0255, 0.8938, 0.6260],
        [0.6369, 0.7785, 0.4601]])

tensor([[-0.3092,  0.3695,  0.5463],
        [ 0.5333, -0.2173,  0.4505]])

tensor([[0.2390, 0.1656, 0.0233],
        [0.0303, 0.1397, 0.0022]])

tensor([[ 0.5367,  2.4096, 14.7179],
        [11.2927,  0.5636, 94.9301]])

tensor([[0.5040, 0.8865, 0.9789],
        [0.9726, 0.5311, 0.9962]])

tensor([[0.3581, 0.1074, 0.0286],
        [0.0152, 0.2806, 0.0045]])


In [200]:
c = torch.tensor([1, -2, 3, -4])
c

tensor([ 1, -2,  3, -4])

In [201]:
# abs
torch.abs(c)

tensor([1, 2, 3, 4])

In [202]:
# negative
torch.neg(c)

tensor([-1,  2, -3,  4])

In [203]:
# Inplace operations
print(torch.abs_(c))
print(c)

print(torch.neg_(c))
print(c)

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([-1, -2, -3, -4])
tensor([-1, -2, -3, -4])


In [204]:
c.fill_(value=10)
# In-place equivalent → c = torch.full_like(c, fill_value=10)

tensor([10, 10, 10, 10])

In [205]:
x = torch.empty_like(input=c)
c.copy_(other=x) # c gets updated, In-place equivalent → c = x.clone()
print(c)

tensor([1276458752,          0,         64,        113])


In [206]:
d = torch.tensor([1.9, 2.3, 3.7, 4.4, 6.5])
d

tensor([1.9000, 2.3000, 3.7000, 4.4000, 6.5000])

In [207]:
# round
torch.round(d) # 6.5 -> 6 (get floored not ceiled)

tensor([2., 2., 4., 4., 6.])

In [208]:
# ceil - nearest bigger integer value
torch.ceil(d)

tensor([2., 3., 4., 5., 7.])

In [209]:
# floor - nearest smaller integer value
torch.floor(d)

tensor([1., 2., 3., 4., 6.])

In [210]:
# clamp | clip - Clipping the values in a certing range
torch.clip(d, min = -1, max = 1)
torch.clamp(d, min = -1, max = 1)

tensor([1., 1., 1., 1., 1.])

In [211]:
e = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float32)
e

tensor([[0., 3., 5.],
        [2., 6., 3.]])

In [212]:
# NumPy `axis` vs Pandas `axis` vs PyTorch `dim` → All are same or might change with function
import pandas as pd

print(np.array(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0))

print(pd.DataFrame(
    [[1, 2, 3], [4, 5, 6]]
).sum(axis=0).to_numpy())

print(torch.tensor(
    [[1, 2, 3], [4, 5, 6]]
).sum(dim=0))

[5 7 9]
[5 7 9]
tensor([5, 7, 9])


In [213]:
# Aggregate operation

# sum
print(torch.sum(e))

# sum along columns
print(torch.sum(e, dim = 0, keepdim=False)) # same as `axis` parameter in numpy

# sum along rows
print(torch.sum(e, dim = 1, keepdim=False))

# keepdim = True
print(torch.sum(e, dim = 1, keepdim=True))

tensor(19.)
tensor([2., 9., 8.])
tensor([ 8., 11.])
tensor([[ 8.],
        [11.]])


*If `keepdim` is `False`, the output tensor will not retain the dimension across which the sum was performed.*

In [214]:
# mean
torch.mean(e)

# mean along col
torch.mean(e, dim=0)

tensor([1.0000, 4.5000, 4.0000])

In [215]:
# median
torch.median(e)

tensor(3.)

In [216]:
# max and min
torch.max(e)
torch.min(e)

tensor(0.)

In [217]:
# product
torch.prod(e) # .mul() is a element-wise multiplication between two matrics

tensor(0.)

In [218]:
# standard deviation
torch.std(e)

tensor(2.1370)

In [219]:
# variance
torch.var(e)

tensor(4.5667)

In [220]:
# argmax
torch.argmax(e)

tensor(4)

In [221]:
# argmin
torch.argmin(e)

tensor(0)

In [222]:
# More _methods()
a = torch.full_like(input=e, fill_value=10)
b = a.clone().copy_(a) * 100
print(a, b, sep="\n")

a.clone().add_(b) # a = a + b (ignore clone)
a.clone().sub_(b) # a = a - b 
a.clone().mul_(b) # a = a * b (element wise)

a.zero_() # x = torch.zeros_like(x)

tensor([[10., 10., 10.],
        [10., 10., 10.]])
tensor([[1000., 1000., 1000.],
        [1000., 1000., 1000.]])


tensor([[0., 0., 0.],
        [0., 0., 0.]])

---

#### Matrix operations

In [223]:
f = torch.randint(size=(2,3), low=0, high=10)
g = torch.randint(size=(3,2), low=0, high=10)

print(f)
print(g)

tensor([[1, 0, 8],
        [8, 8, 7]])
tensor([[1, 2],
        [8, 1],
        [6, 9]])


In [224]:
# matrix multiplcation - dot product
torch.matmul(f, g)

tensor([[ 49,  74],
        [114,  87]])

In [225]:
vector1 = torch.tensor([1, 2])
vector2 = torch.tensor([3, 4])

# dot product - only works with 1D Tensors
torch.dot(vector1, vector2)

tensor(11)

In [226]:
# transpose
torch.transpose(input=f, dim0=0, dim1=1)

tensor([[1, 8],
        [0, 8],
        [8, 7]])

In [227]:
torch.transpose(f, dim0 = 1, dim1 = 0) # Transpose is same as `reshape` operation by swapping the dimentions

tensor([[1, 8],
        [0, 8],
        [8, 7]])

In [228]:
# 0th axis - 1, 1st axis - 2, 2nd axis - 3 → consider size as a list and axis is an index
torch.rand(size=[1, 2, 3,]).transpose(dim0=0, dim1=2).shape

torch.Size([3, 2, 1])

In [229]:
# Implace Transpose → only works with 2D
f.t_()

tensor([[1, 8],
        [0, 8],
        [8, 7]])

In [230]:
temp = torch.tensor([
    [[1], [2]],
    [[1], [3]],
    [[1], [4]],
])

print(temp.shape)
torch.transpose(temp, dim0=0, dim1=1).shape # Check the dimentions and its index correctly

torch.Size([3, 2, 1])


torch.Size([2, 3, 1])

*Tip: To visualize a transpose, flatten the tensor into a 1D array and then re-arrange the elements according to the target shape.*

*For a tensor with shape [2, 3, 1], the 0th dimension has size 2, the 1st dimension has size 3, and so on.*

In [231]:
# Convert any nD tensor to 1D tensor
print(torch.tensor([[1, 2], [3, 4]]).ravel())
print(torch.tensor([[1, 2], [3, 4]]).flatten())
print(torch.tensor([[[1], [2]], [[3], [4]]]).reshape(-1))

tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])
tensor([1, 2, 3, 4])


In [232]:
h = torch.randint(size=(3,3), low=0, high=10, dtype=torch.float32)
h

tensor([[0., 2., 9.],
        [4., 1., 5.],
        [8., 5., 4.]])

In [233]:
# determinant
torch.det(h)

tensor(156.)

In [234]:
# inverse
torch.inverse(h)

tensor([[-0.1346,  0.2372,  0.0064],
        [ 0.1538, -0.4615,  0.2308],
        [ 0.0769,  0.1026, -0.0513]])

---

#### Comparison operations

In [235]:
i = torch.randint(size=(2,3), low=0, high=10)
j = torch.randint(size=(2,3), low=0, high=10)

print(i)
print(j)

tensor([[8, 3, 8],
        [2, 7, 3]])
tensor([[7, 0, 5],
        [7, 8, 4]])


In [236]:
# Element wise comparison - Shape must be same or able to broadcast

# greater than
print(i > j)

# less than
print(i < j)

# equal to
print(i == j)

# not equal to
print(i != j)

tensor([[ True,  True,  True],
        [False, False, False]])
tensor([[False, False, False],
        [ True,  True,  True]])
tensor([[False, False, False],
        [False, False, False]])
tensor([[True, True, True],
        [True, True, True]])


In [237]:
# Boolean Indexing - Reshaped into 1D
i[i > j]

tensor([8, 3, 8])

---

#### Special functions

In [238]:
k = torch.randint(size=(2,3), low=0, high=10, dtype=torch.float64)
k

tensor([[1., 8., 9.],
        [7., 6., 5.]], dtype=torch.float64)

In [239]:
# log - base e
torch.log(input = k)

# log - base 10
torch.log10(input = k)

tensor([[0.0000, 0.9031, 0.9542],
        [0.8451, 0.7782, 0.6990]], dtype=torch.float64)

In [240]:
# Checking the size of the elements in the tensor
tensor = torch.log10(input = k)
print(tensor.dtype)
tensor.element_size()  # size in bytes of a single element

torch.float64


8

In [241]:
# exp
torch.exp(k)

tensor([[2.7183e+00, 2.9810e+03, 8.1031e+03],
        [1.0966e+03, 4.0343e+02, 1.4841e+02]], dtype=torch.float64)

In [242]:
# sqrt
torch.sqrt(k)

tensor([[1.0000, 2.8284, 3.0000],
        [2.6458, 2.4495, 2.2361]], dtype=torch.float64)

In [243]:
# sigmoid
torch.sigmoid(k)

tensor([[0.7311, 0.9997, 0.9999],
        [0.9991, 0.9975, 0.9933]], dtype=torch.float64)

In [244]:
# softmax
torch.softmax(k, dim = 1)

tensor([[2.4518e-04, 2.6888e-01, 7.3088e-01],
        [6.6524e-01, 2.4473e-01, 9.0031e-02]], dtype=torch.float64)

In [245]:
# relu
torch.relu(k)

tensor([[1., 8., 9.],
        [7., 6., 5.]], dtype=torch.float64)

---

#### Inplace Operations

In [246]:
m = torch.rand(2,3)
n = torch.rand(2,3)

print(m)
print(n)

tensor([[0.9758, 0.5092, 0.3043],
        [0.6345, 0.1077, 0.3228]])
tensor([[0.4300, 0.6588, 0.4684],
        [0.0516, 0.5151, 0.4416]])


In [247]:
# _ methods
m.add_(n) # It returns the new matrix

tensor([[1.4057, 1.1680, 0.7727],
        [0.6861, 0.6228, 0.7643]])

In [248]:
m # Updated

tensor([[1.4057, 1.1680, 0.7727],
        [0.6861, 0.6228, 0.7643]])

In [249]:
n

tensor([[0.4300, 0.6588, 0.4684],
        [0.0516, 0.5151, 0.4416]])

In [250]:
torch.relu(m)

tensor([[1.4057, 1.1680, 0.7727],
        [0.6861, 0.6228, 0.7643]])

In [251]:
m.relu_()

tensor([[1.4057, 1.1680, 0.7727],
        [0.6861, 0.6228, 0.7643]])

In [252]:
m

tensor([[1.4057, 1.1680, 0.7727],
        [0.6861, 0.6228, 0.7643]])

---

#### Copying a Tensor

In [253]:
a = torch.tensor(
    data = [[1,2,3], [4,5,6]]
)
b = a.clone() # Deep Copy
c = a._lazy_clone() # Shallow Copy → NOTE: If you make any changes in either a or c then those changes will be reflacted in the other one

print(a, b, c, sep = '\n')

print(id(a), id(b), id(c))
print(id(a[0,0]), id(b[0][0]), id(c[0][0]))

tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
tensor([[1, 2, 3],
        [4, 5, 6]])
139244370530640 139244370531536 139244370109264
139244370531664 139244370528464 139244370531664


In [254]:
a[0][0] = 10

print(a, b, sep = '\n')

tensor([[10,  2,  3],
        [ 4,  5,  6]])
tensor([[1, 2, 3],
        [4, 5, 6]])


In [255]:
id(a)

139244370530640

In [256]:
id(b)

139244370531536

---

A third-order polynomial, trained to predict `y = sin(x) from -pi to pi` by minimizing squared Euclidean distance.

In [257]:
# Variables Initialization
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float
learning_rate = 1e-6

# Dataset
x = torch.linspace(start=-torch.pi, end=torch.pi, steps=2000) # shape -> 2000 1D
y = torch.sin(input=x) # Shape -> 2000 1D

# Random Weights → for 3 degree term | shape -> Scaler
a = torch.randn((), dtype=dtype)
b = torch.randn((), dtype=dtype)
c = torch.randn((), dtype=dtype)
d = torch.randn((), dtype=dtype)

def y_pred_equation(X_train):
    y_pred = a + (b * x) + (c * x.pow(2)) + (d * x.pow(3))
    return y_pred # shape -> 2000 1D

def main():
    # Training
    for i in range(2500):
        # Prediction
        y_pred = y_pred_equation(x)

        # Loss Calculation
        loss = (y - y_pred).pow(2).sum().item()

        if i % 100 == 99:
            print(f"loss at {i + 1}it epoch → {loss}")

        # Gradients Calculation
        grad_y_pred = -2 * (y - y_pred) # shape -> 2000
        grad_a = grad_y_pred.sum()
        grad_b = torch.matmul(input = grad_y_pred, other = x).sum()
        grad_c = torch.matmul(input = grad_y_pred, other = x.pow(2)).sum()
        grad_d = torch.matmul(input = grad_y_pred, other = x.pow(3)).sum()

        # Updating
        global a, b, c, d
        a = a - learning_rate * grad_a
        b -= learning_rate * grad_b
        c -= learning_rate * grad_c
        d -= learning_rate * grad_d

main()

loss at 100it epoch → 3208.857421875
loss at 200it epoch → 2125.1279296875
loss at 300it epoch → 1408.42919921875
loss at 400it epoch → 934.453369140625
loss at 500it epoch → 620.9949951171875
loss at 600it epoch → 413.6910705566406
loss at 700it epoch → 276.590576171875
loss at 800it epoch → 185.91802978515625
loss at 900it epoch → 125.95044708251953
loss at 1000it epoch → 86.28941345214844
loss at 1100it epoch → 60.05842590332031
loss at 1200it epoch → 42.70944595336914
loss at 1300it epoch → 31.23479652404785
loss at 1400it epoch → 23.64527130126953
loss at 1500it epoch → 18.6254940032959
loss at 1600it epoch → 15.305182456970215
loss at 1700it epoch → 13.108964920043945
loss at 1800it epoch → 11.656250953674316
loss at 1900it epoch → 10.695302963256836
loss at 2000it epoch → 10.059648513793945
loss at 2100it epoch → 9.639156341552734
loss at 2200it epoch → 9.36098575592041
loss at 2300it epoch → 9.176968574523926
loss at 2400it epoch → 9.05522632598877
loss at 2500it epoch → 8.9746